In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install requests pandas pyarrow -q
# import pmcid_backfill

In [ ]:
#!/usr/bin/env python3
"""
pmcid_backfill.py  —  Google Colab
==================================
Free coverage win: resolve DOI/PMID -> PMCID via NCBI's PMC ID Converter for the
studies where OpenAlex carried no PMCID. Every failure chain in your harvest run
starts with `no_pmcid>no_pmcid>...`, so filling this column converts a slice of
"no route" studies into "Europe PMC / PMC works" — the highest-quality source,
with real JATS section tags (data-availability, ethics, CRediT).

WHY THIS WORKS: OpenAlex does not always carry the PMCID even when PMC has the
article. The ID Converter maps DOI/PMID/PMCID both ways, 200 ids per request,
keyless (an NCBI API key raises the rate limit 3 -> 10 rps).

OUTPUT
  pmcid_backfill.parquet   oa_id, oa_doi, oa_pmid, pmcid_found, source_id_used
  and (optionally) rewrites your corpus parquet with oa_pmcid filled in.

AFTER RUNNING: rerun the harvester. Studies that gained a PMCID will now take the
Europe PMC / PMC branch instead of failing.

SETUP
  !pip install requests pandas pyarrow -q
"""

import json, sys, time
from pathlib import Path
import pandas as pd
import requests

# =========================================================================== #
# CONFIG                                                                        #
# =========================================================================== #
IN_PARQUET   = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/llm_screen/screened_llm.parquet"
OUT_MAP      = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/pmcid_backfill.parquet"
OUT_PARQUET  = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/with_pmcid.parquet"  # corpus + filled PMCIDs
WRITE_CORPUS = True

MAILTO   = "cognosxx@gmail.com"
NCBI_KEY = ""            # optional: 3 -> 10 requests/sec
BATCH    = 200           # ID Converter hard max per request
SLEEP    = 0.34          # ~3 rps without a key; drop to 0.1 with one

CONV_URL = "https://pmc.ncbi.nlm.nih.gov/tools/idconv/api/v1/articles/"

SESS = requests.Session()
SESS.headers.update({"User-Agent": f"tick-sos-pmcid/1.0 (mailto:{MAILTO})"})


def S(x):
    return "" if x is None or (isinstance(x, float) and pd.isna(x)) or x is pd.NA else str(x)
def norm_doi(x):
    return S(x).lower().replace("https://doi.org/", "").replace("doi.org/", "").strip()
def clean_pmid(x):
    v = S(x).strip()
    return v if v.isdigit() else ""


# =========================================================================== #
# batch convert                                                                #
# =========================================================================== #
def convert(ids, idtype):
    """ids: list of DOIs or PMIDs. Returns {input_id: pmcid}."""
    out = {}
    params = {"ids": ",".join(ids), "format": "json", "email": MAILTO, "idtype": idtype}
    if NCBI_KEY:
        params["api_key"] = NCBI_KEY
    for attempt in range(4):
        try:
            r = SESS.get(CONV_URL, params=params, timeout=60)
            if r.status_code == 200:
                try:
                    recs = r.json().get("records", [])
                except json.JSONDecodeError:
                    return out
                for rec in recs:
                    key = S(rec.get(idtype) or rec.get("doi") or rec.get("pmid")).lower()
                    pmcid = S(rec.get("pmcid"))
                    if key and pmcid:
                        out[key] = pmcid
                return out
            if r.status_code in (429, 500, 502, 503):
                time.sleep(2 ** attempt); continue
            return out
        except requests.RequestException:
            time.sleep(2 ** attempt)
    return out


def chunks(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i + n]


# =========================================================================== #
if __name__ == "__main__":
    df_all = pd.read_parquet(IN_PARQUET)
    df = df_all[df_all["llm_label"].notna()]
    print(f"[in] {len(df):,} rows", file=sys.stderr)

    has_pmcid = df["oa_pmcid"].map(lambda x: bool(S(x))) if "oa_pmcid" in df.columns \
                else pd.Series([False] * len(df))
    todo = df[~has_pmcid].copy()
    print(f"[target] {len(todo):,} studies without a PMCID "
          f"({has_pmcid.sum():,} already have one)", file=sys.stderr)

    todo["_doi"] = todo["oa_doi"].map(norm_doi) if "oa_doi" in todo.columns else ""
    todo["_pmid"] = todo["oa_pmid"].map(clean_pmid) if "oa_pmid" in todo.columns else ""

    # --- pass 1: by DOI (largest pool) ---
    dois = sorted({d for d in todo["_doi"] if d})
    print(f"[pass1] {len(dois):,} distinct DOIs -> ID Converter", file=sys.stderr)
    doi_map = {}
    for i, batch in enumerate(chunks(dois, BATCH), 1):
        doi_map.update(convert(batch, "doi"))
        time.sleep(SLEEP)
        if i % 25 == 0:
            print(f"[pass1] {i * BATCH:,}/{len(dois):,} | found {len(doi_map):,}",
                  file=sys.stderr)
    print(f"[pass1] resolved {len(doi_map):,} PMCIDs from DOIs", file=sys.stderr)

    # --- pass 2: by PMID for those still unresolved ---
    unresolved = todo[~todo["_doi"].isin(doi_map.keys())]
    pmids = sorted({p for p in unresolved["_pmid"] if p})
    print(f"[pass2] {len(pmids):,} distinct PMIDs -> ID Converter", file=sys.stderr)
    pmid_map = {}
    for i, batch in enumerate(chunks(pmids, BATCH), 1):
        pmid_map.update(convert(batch, "pmid"))
        time.sleep(SLEEP)
        if i % 25 == 0:
            print(f"[pass2] {i * BATCH:,}/{len(pmids):,} | found {len(pmid_map):,}",
                  file=sys.stderr)
    print(f"[pass2] resolved {len(pmid_map):,} PMCIDs from PMIDs", file=sys.stderr)

    # --- assemble ---
    rows = []
    for _, r in todo.iterrows():
        d, p = r["_doi"], r["_pmid"]
        pmcid, via = "", ""
        if d and d in doi_map:
            pmcid, via = doi_map[d], "doi"
        elif p and p in pmid_map:
            pmcid, via = pmid_map[p], "pmid"
        rows.append({"oa_id": S(r.get("oa_id")), "oa_doi": d, "oa_pmid": p,
                     "pmcid_found": pmcid, "source_id_used": via})
    m = pd.DataFrame(rows)
    m.to_parquet(OUT_MAP, index=False)
    gained = (m["pmcid_found"] != "").sum()
    print(f"\n[gain] {gained:,} new PMCIDs "
          f"({gained/max(len(m),1):.1%} of the no-PMCID pool)", file=sys.stderr)
    print(f"[write] {OUT_MAP}", file=sys.stderr)

    if WRITE_CORPUS and gained:
        fill = dict(zip(m["oa_id"], m["pmcid_found"]))
        if "oa_pmcid" not in df.columns:
            df["oa_pmcid"] = ""
        df["oa_pmcid"] = [
            S(pm) if S(pm) else fill.get(S(oid), "")
            for oid, pm in zip(df["oa_id"], df["oa_pmcid"])
        ]
        df.to_parquet(OUT_PARQUET, index=False)
        now = df["oa_pmcid"].map(lambda x: bool(S(x))).sum()
        print(f"[write] {OUT_PARQUET}  (PMCID coverage now {now:,}/{len(df):,} "
              f"= {now/len(df):.1%})", file=sys.stderr)
        print("[next] point the harvester's IN_PARQUET at this file and rerun — "
              "newly-resolved studies will take the Europe PMC / PMC branch.",
              file=sys.stderr)

In [ ]:
df

In [ ]:
m